In [33]:
from datasets import load_dataset
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from transformers import (
    BertTokenizer,
    BertModel
)
from torch.nn.functional import pad
import torch.nn as nn
from tqdm import tqdm


In [74]:
ds_train = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train")
ds_test = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
print(ds_train[3]) 
print(ds_train[10])
print(ds_train[12])  
print(ds_train[20]) 

{'text': '$ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N', 'label': 0}
{'text': "$HOG - Moody's warns on Harley-Davidson https://t.co/LurHBEadeU", 'label': 0}
{'text': '$I - Intelsat cut to Market Perform at Raymond James https://t.co/YsvsMSQRIb', 'label': 0}
{'text': '$NCBS: Hovde Group cuts to Market Perform', 'label': 0}


https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment?utm_source=chatgpt.com

In [3]:
print(ds_train[3])
print(ds_train[10])
print(ds_train[-2]) 

sentiments = {
    "0": "Bearish", 
    "1": "Bullish", 
    "2": "Neutral"
}  

x_train, y_train = [x["text"] for x in ds_train], [x["label"] for x in ds_train] 
x_test, y_test   = [x["text"] for x in ds_test], [x["label"] for x in ds_test] 




{'text': '$ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N', 'label': 0}
{'text': "$HOG - Moody's warns on Harley-Davidson https://t.co/LurHBEadeU", 'label': 0}
{'text': 'WORK, XPO, PYX and AMKR among after hour movers', 'label': 2}


Average Tweet size

In [80]:
size = 0
for i in range(len(x_train)):
    x = x_train[i]
    words = x.split()
    size += len(words)

print(f"Average tweet size {size / len(x_train)} words")

Average tweet size 12.17835062349366 words


In [4]:
class FinancialDataset(Dataset):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    
    def __init__(self, x_data, y_data):
        self.x_data = x_data
        self.y_data = y_data

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        tokens = self.tokenizer(self.x_data[idx], return_tensors="pt")
        return tokens.input_ids, self.y_data[idx]

db_train = FinancialDataset(x_train, y_train)
db_test  = FinancialDataset(x_test, y_test)

print(db_train[0][0].shape)

torch.Size([1, 32])


In [23]:
def merge_batch(xs):
    inputs_shape = [x[0].shape[1] for x in xs] 
    m = max(inputs_shape)
    pad_nums = [m - x[0].shape[1] for x in xs]
    input_tokens = [pad(x[0], (0, pad_nums[i])) for i, x in enumerate(xs)]
    targets = [x[1] for x in xs]
    input_batch = torch.stack(input_tokens)
    attention_mask = [torch.cat((torch.ones(x), torch.zeros(pad_nums[i])), dim=0) for i, x in enumerate(inputs_shape)]
    attention_mask = torch.stack(attention_mask)
    return input_batch.squeeze(1), attention_mask, torch.tensor(targets)


train_loader = DataLoader(
    db_train,
    batch_size=3,
    shuffle=True,
    collate_fn=merge_batch
)

test_loader = DataLoader(
    db_test,
    batch_size=16,
    shuffle=False
)

input, att, out = next(iter(train_loader))
print(input.shape, att.shape, out)

torch.Size([3, 46]) torch.Size([3, 46]) tensor([2, 2, 1])


In [6]:
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3532.49it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
input, att, out = next(iter(train_loader))
print(input.shape, att.shape)
hs = model(input_ids=input, 
           attention_mask=att).last_hidden_state
print(hs.shape)
hs[:,0,:]

torch.Size([3, 32]) torch.Size([3, 32])
torch.Size([3, 32, 768])


tensor([[-0.2359, -0.3601, -0.2600,  ..., -0.5817,  0.1513,  0.5556],
        [ 0.0895, -0.2146, -0.1855,  ...,  0.0651,  0.2541,  0.3509],
        [-0.8955, -0.4794, -0.0844,  ..., -0.3259,  0.5714,  0.1115]],
       grad_fn=<SelectBackward0>)

### Define a classifier

CLS token

In [18]:
class Classifier(nn.Module):
    model_name = "bert-base-uncased"

    def __init__(self, num_classes, input_size=768):
        super().__init__()
        self.model = BertModel.from_pretrained(self.model_name)
        self.classifier = nn.Linear(
            input_size,
            num_classes
        )

    def forward(self, x, attention_mask):
        hs = self.model(input_ids=x, 
                   attention_mask=attention_mask).last_hidden_state

        cls_embedding = hs[:,0,:]
        return self.classifier(cls_embedding)


cls = Classifier(3)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5298.36it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
input, att, out = next(iter(train_loader))
print(input.shape)
cls(input,att).shape

torch.Size([18, 51])


torch.Size([18, 3])

# Training loop

In [35]:
train_loader = DataLoader(
    db_train,
    batch_size=18,
    shuffle=True,
    collate_fn=merge_batch
)

test_loader = DataLoader(
    db_test,
    batch_size=18,
    shuffle=True,
    collate_fn=merge_batch
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Classifier(3)
model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4996.27it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
def evaluate(model, test_loader):
    with torch.no_grad():
        model.eval()
        total_loss = 0
        correct = 0
        total = 0
        for inputs, attention_mask, targets in tqdm(test_loader, desc="Test"):
            inputs = inputs.to(device)
            attention_mask = attention_mask.to(device)
            targets = targets.to(device)
            logits = model(inputs, attention_mask)
            loss = criterion(logits, targets)

            # Statistics
            total_loss += loss.item()
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)

        avg_loss = total_loss / len(train_loader)
        accuracy = correct / total
    print(f"Eval Loss: {avg_loss:.4f} " f"Accuracy: {accuracy:.4f}")

evaluate(model, test_loader)

Test: 100%|██████████| 133/133 [00:07<00:00, 18.92it/s]

Eval Loss: 0.2622 Accuracy: 0.4946


In [38]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for inputs, attention_mask, targets in tqdm(train_loader, desc="Training"):
        inputs = inputs.to(device)
        attention_mask = attention_mask.to(device)
        targets = targets.to(device)
        # Clear gradients
        optimizer.zero_grad()
        # Forward pass
        logits = model(inputs, attention_mask)
        # Calculate loss
        loss = criterion(logits, targets)
        # Backpropagation
        loss.backward()
        # Update parameters
        optimizer.step()

        # Statistics
        total_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"Loss: {avg_loss:.4f} "
        f"Accuracy: {accuracy:.4f}"
    )
    evaluate(model, test_loader)


Training: 100%|██████████| 531/531 [01:52<00:00,  4.71it/s]


Epoch 1/3 Loss: 0.5080 Accuracy: 0.7979


Test: 100%|██████████| 133/133 [00:07<00:00, 18.91it/s]


Eval Loss: 0.0903 Accuracy: 0.8564


Training: 100%|██████████| 531/531 [01:53<00:00,  4.69it/s]


...

Eval Loss: 0.0836 Accuracy: 0.8869


Training: 100%|██████████| 531/531 [01:52<00:00,  4.73it/s]


Epoch 3/3 Loss: 0.1162 Accuracy: 0.9581


Test: 100%|██████████| 133/133 [00:06<00:00, 19.06it/s]

Eval Loss: 0.1077 Accuracy: 0.8727


In [ ]:
evaluate(model, test_loader)



Test: 100%|██████████| 133/133 [00:07<00:00, 18.87it/s]

Eval Loss: 0.1078 Accuracy: 0.8727


## Production

In [71]:
def predict(model, text):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    tokens = tokenizer(text, return_tensors="pt")
    input_ids = tokens["input_ids"].to("cuda")
    attention_mask = tokens["attention_mask"].to("cuda")

    model.eval()
    with torch.no_grad():
        logits = model(
            x=input_ids,
            attention_mask=attention_mask
        )

    prediction = torch.argmax(logits, dim=1).item()
    return sentiments[str(prediction)]


In [70]:
texts = [
    "The company reported stronger earnings.",
    "The company reported higher-than-expected profits.",
    "Revenue increased significantly during the quarter.",
    "The company achieved record sales this year.",
    "Operating profit improved compared with last year.",
    "The company raised its earnings forecast.",
    "Strong demand boosted the company's quarterly results.",
    "The firm reported solid growth in revenue.",
    "Profit margins improved significantly during the quarter.",
    "The company exceeded analysts' expectations."
]
for text in texts:
    out = predict(model, text)
    print(out)

Bullish
Bullish
Bullish
Bullish
Bullish
Bullish
Bullish
Bullish
Bullish
Bullish


In [72]:
bearish_texts = [
    "The company reported significantly weaker earnings than expected.",
    "The firm's profits fell sharply during the quarter.",
    "Revenue declined substantially compared with last year.",
    "The company lowered its full-year earnings forecast.",
    "Weak demand is expected to weigh on future sales.",
    "The stock fell after the company reported disappointing results.",
    "Analysts expect the company's earnings to deteriorate further.",
    "Rising costs put significant pressure on profit margins.",
    "The company missed revenue expectations for the second consecutive quarter.",
    "Management warned that challenging market conditions could reduce profits.",
    "The firm expects lower sales in the coming quarters.",
    "The company's debt increased while cash flow weakened.",
    "Investors reacted negatively to the disappointing earnings report.",
    "The company faces declining demand and increasing competition.",
    "Profit margins contracted significantly during the quarter."
]

for text in bearish_texts:
    out = predict(model, text)
    print(out)

Bearish
Bearish
Neutral
Bearish
Bearish
Bearish
Bearish
Bearish
Bearish
Bearish
Bearish
Bearish
Bearish
Bearish
Bearish


In [81]:
neutral_texts = [
    "The company reported its quarterly earnings on Tuesday.",
    "The company announced its financial results for the third quarter.",
    "Revenue was €2.4 billion during the quarter.",
    "The company plans to open a new production facility next year.",
    "Management discussed the company's financial performance.",
    "The firm announced changes to its executive team.",
    "The company maintained its current earnings forecast.",
    "The company released its annual financial report.",
    "The firm plans to invest €50 million in new equipment.",
    "The company expects to publish its next earnings report in October.",
    "The board approved the company's annual financial statements.",
    "The company signed a new agreement with a European supplier.",
    "The firm currently operates production facilities in five countries.",
    "The company reported quarterly revenue of €850 million.",
    "Management said the company will continue its current investment strategy."
]

for text in neutral_texts:
    out = predict(model, text)
    print(out)

Neutral
Neutral
Bullish
Neutral
Neutral
Neutral
Bullish
Neutral
Neutral
Neutral
Neutral
Neutral
Neutral
Bullish
Neutral
